# 11 — Predicting real text embeddings

This lesson replaces synthetic text-like targets with real CIFAR-10 class prompts. A small predictor maps frozen image features to text embeddings. A linear classifier is the baseline.

## Learning goals

- Explain how an image model can predict an embedding of a text target.
- Encode real class prompts and rank them by similarity to a predicted embedding.
- Compare the result with a linear classifier on the same frozen image features.
- Separate this small local experiment from the VL-JEPA paper's findings.

## What this lesson is

This is a **small supervised analogue** of embedding prediction. It is not a VL-JEPA reproduction, zero-shot classification, or open-vocabulary evaluation. The image labels are used to train both methods. The visual backbone was pretrained on ImageNet.

The paper reports a 1.6B-parameter vision-language system. Its base model uses V-JEPA 2 ViT-L, a predictor initialized from Llama-3.2-1B, and EmbeddingGemma. It trains on billions of image and video examples and reports results on eight video-classification and eight text-to-video retrieval datasets. See the [primary paper](https://arxiv.org/html/2512.10942).

## Requirements and downloads

Use Python 3.10–3.13, the project PyTorch/TorchVision install, and the `vljepa` optional dependency extra. See [the quickstart](../specs/005-vljepa-real-labels/quickstart.md).

The first run needs network access and downloads about 300 MB: CIFAR-10 (163 MB), ResNet-18 weights (44.7 MB), and MiniLM weights (90.9 MB). Later offline runs need these assets in the local caches. CPU is supported; a GPU is optional.

The [CIFAR-10 source page](https://www.cs.toronto.edu/~kriz/cifar.html) asks users to cite its technical report but does not state license terms. TorchVision warns that model weights can have separate use terms. Review terms for your use. This repository does not include or redistribute these files.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from jepa_tutorial.storage import DATA_DIR
import platform
import random
import time

import numpy as np
import scipy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import transformers
from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision import datasets
from torchvision.models import ResNet18_Weights, resnet18
from transformers import AutoModel, AutoTokenizer

assert int(transformers.__version__.split(".")[0]) >= 5, "Install the optional vljepa dependency extra."

DATA_SEED = 20261014
MODEL_SEEDS = [7, 17, 27, 37, 47]
SAMPLES_PER_CLASS = 100
FEATURE_BATCH_SIZE = 64
HEAD_BATCH_SIZE = 128
EPOCHS = 25
LEARNING_RATE = 1e-3

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(DATA_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    torch.set_num_threads(min(4, torch.get_num_threads()))

hardware = torch.cuda.get_device_name(0) if device.type == "cuda" else platform.processor() or platform.machine()
print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__}; SciPy: {scipy.__version__}")
print(f"PyTorch: {torch.__version__}; TorchVision: {torchvision.__version__}; Transformers: {transformers.__version__}")
print(f"Platform: {platform.platform()}")
print(f"Device: {device} ({hardware})")
print(f"Data seed: {DATA_SEED}; model seeds: {MODEL_SEEDS}")
print(f"Training settings: {SAMPLES_PER_CLASS} samples/class, {EPOCHS} epochs, feature batch {FEATURE_BATCH_SIZE}, head batch {HEAD_BATCH_SIZE}, learning rate {LEARNING_RATE}, weight decay 1e-4")


## 1. Encode ten real text targets

Each target is a fixed natural-language prompt for one CIFAR-10 class. The text encoder is frozen. We use the model card's attention-mask mean pooling and L2 normalization.

The model is `sentence-transformers/all-MiniLM-L6-v2`, pinned to revision `ea78891063587eb050ed4166b20062eaf978037c`. Its model card lists Apache-2.0, a 384-dimensional output, and a 90.9 MB model file. This is a general sentence-embedding model, not the paper's EmbeddingGemma text encoder.


In [ ]:
TEXT_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
TEXT_MODEL_REVISION = "ea78891063587eb050ed4166b20062eaf978037c"

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]
articles = {"airplane": "an", "automobile": "an"}
prompts = [f"A photo of {articles.get(name, 'a')} {name}." for name in class_names]

tokenizer = AutoTokenizer.from_pretrained(
    TEXT_MODEL_ID,
    revision=TEXT_MODEL_REVISION,
)
text_encoder = AutoModel.from_pretrained(
    TEXT_MODEL_ID,
    revision=TEXT_MODEL_REVISION,
    use_safetensors=True,
).to("cpu").eval()

tokens = tokenizer(prompts, padding=True, truncation=True, return_tensors="pt")
with torch.inference_mode():
    token_embeddings = text_encoder(**tokens).last_hidden_state
    mask = tokens["attention_mask"].unsqueeze(-1).to(token_embeddings.dtype)
    text_embeddings = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
    text_embeddings = F.normalize(text_embeddings, p=2, dim=1)

assert text_embeddings.shape == (10, 384), text_embeddings.shape
assert torch.isfinite(text_embeddings).all()
assert torch.allclose(text_embeddings.norm(dim=1), torch.ones(10), atol=1e-5)

for name, prompt in zip(class_names, prompts):
    print(f"{name:>10}: {prompt}")
print(f"Text embedding matrix: {tuple(text_embeddings.shape)}; all rows are normalized.")


## 2. Load a fixed, balanced CIFAR-10 sample

CIFAR-10 has 50,000 training images and 10,000 test images. This lesson selects 100 images per class from each official split. It does not move images between the official train and test sets.

TorchVision downloads CIFAR-10 to `notebooks/data/` by default. Set `JEPA_ASSETS_DIR` before starting Jupyter to use a different asset root. It uses the preprocessing attached to the named ResNet-18 weights: resize, center crop, and ImageNet normalization.


In [ ]:
image_weights = ResNet18_Weights.IMAGENET1K_V1
image_transform = image_weights.transforms()
train_dataset = datasets.CIFAR10(
    root=str(DATA_DIR),
    train=True,
    download=True,
    transform=image_transform,
)
test_dataset = datasets.CIFAR10(
    root=str(DATA_DIR),
    train=False,
    download=True,
    transform=image_transform,
)

assert train_dataset.classes == class_names, train_dataset.classes

def balanced_indices(targets, n_per_class, seed):
    rng = np.random.default_rng(seed)
    selected = []
    targets = np.asarray(targets)
    for class_id in range(len(class_names)):
        candidates = np.flatnonzero(targets == class_id)
        if len(candidates) < n_per_class:
            raise ValueError(f"Class {class_id} has {len(candidates)} examples; need {n_per_class}.")
        selected.extend(np.sort(rng.choice(candidates, n_per_class, replace=False)).tolist())
    return selected

train_indices = balanced_indices(train_dataset.targets, SAMPLES_PER_CLASS, DATA_SEED)
test_indices = balanced_indices(test_dataset.targets, SAMPLES_PER_CLASS, DATA_SEED + 1)
train_loader = DataLoader(Subset(train_dataset, train_indices), batch_size=FEATURE_BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(Subset(test_dataset, test_indices), batch_size=FEATURE_BATCH_SIZE, shuffle=False, num_workers=0)

train_counts = np.bincount(np.asarray(train_dataset.targets)[train_indices], minlength=10)
test_counts = np.bincount(np.asarray(test_dataset.targets)[test_indices], minlength=10)
assert np.array_equal(train_counts, np.full(10, SAMPLES_PER_CLASS))
assert np.array_equal(test_counts, np.full(10, SAMPLES_PER_CLASS))
assert len(set(train_indices)) == len(train_indices)
assert len(set(test_indices)) == len(test_indices)
print(f"Training counts: {train_counts.tolist()}")
print(f"Held-out counts: {test_counts.tolist()}")
print("The two subsets come from separate official CIFAR-10 splits.")


## 3. Extract frozen visual features

We use the official TorchVision `ResNet18_Weights.IMAGENET1K_V1` weights. The documented file size is 44.7 MB. The weights use ImageNet supervision, so this is not a test of vision-only self-supervised learning.

TorchVision does not state a specific license for these weights in the model page. Its repository says individual model weights can have separate terms. Check the source terms for your use.


In [ ]:
image_encoder = resnet18(weights=image_weights)
image_encoder.fc = nn.Identity()
image_encoder = image_encoder.to(device).eval()

def extract_features(loader, encoder):
    feature_parts = []
    label_parts = []
    with torch.inference_mode():
        for images, labels in loader:
            features = encoder(images.to(device)).flatten(start_dim=1)
            feature_parts.append(features.cpu())
            label_parts.append(labels.cpu())
    return torch.cat(feature_parts), torch.cat(label_parts)

feature_start = time.perf_counter()
train_features, train_labels = extract_features(train_loader, image_encoder)
test_features, test_labels = extract_features(test_loader, image_encoder)
feature_seconds = time.perf_counter() - feature_start

assert train_features.shape == (1000, 512)
assert test_features.shape == (1000, 512)
assert torch.isfinite(train_features).all() and torch.isfinite(test_features).all()
print(f"Training features: {tuple(train_features.shape)}")
print(f"Held-out features: {tuple(test_features.shape)}")
print(f"Feature extraction time: {feature_seconds:.1f} seconds")


## 4. Train the predictor and the baseline

The predictor learns to map a 512-dimensional image feature to the normalized 384-dimensional embedding for its known class prompt. At test time, it ranks the ten prompts by cosine similarity.

The baseline is a linear ten-class classifier trained with cross-entropy. It uses the same frozen image features and exactly the same examples. Both methods use the same five initialization seeds. The heads have different output sizes and objectives, so this is a practical baseline comparison, not a capacity-matched study.


In [ ]:
class TextEmbeddingPredictor(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Linear(256, 384),
        )

    def forward(self, features):
        return self.layers(features)


def train_text_predictor(seed):
    torch.manual_seed(seed)
    model = TextEmbeddingPredictor()
    targets = text_embeddings[train_labels]
    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader(
        TensorDataset(train_features, targets),
        batch_size=HEAD_BATCH_SIZE,
        shuffle=True,
        generator=generator,
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    for _ in range(EPOCHS):
        model.train()
        for features, target in loader:
            predicted = F.normalize(model(features), dim=1)
            loss = 1 - (predicted * target).sum(dim=1).mean()
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite text-prediction loss.")
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
    model.eval()
    with torch.inference_mode():
        predicted = F.normalize(model(test_features), dim=1)
        scores = predicted @ text_embeddings.T
        predictions = scores.argmax(dim=1)
    return predictions, scores


def train_linear_classifier(seed):
    torch.manual_seed(seed)
    model = nn.Linear(512, 10)
    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader(
        TensorDataset(train_features, train_labels),
        batch_size=HEAD_BATCH_SIZE,
        shuffle=True,
        generator=generator,
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    for _ in range(EPOCHS):
        model.train()
        for features, labels in loader:
            logits = model(features)
            loss = F.cross_entropy(logits, labels)
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite classifier loss.")
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
    model.eval()
    with torch.inference_mode():
        scores = model(test_features)
        predictions = scores.argmax(dim=1)
    return predictions, scores


def run_method(train_fn):
    runs = []
    for seed in MODEL_SEEDS:
        predictions, scores = train_fn(seed)
        assert predictions.shape == test_labels.shape
        assert torch.isfinite(scores).all()
        correct = predictions.eq(test_labels)
        per_class = []
        for class_id in range(10):
            class_mask = test_labels.eq(class_id)
            per_class.append(correct[class_mask].float().mean().item())
        runs.append({
            "seed": seed,
            "accuracy": correct.float().mean().item(),
            "per_class_accuracy": per_class,
            "predictions": predictions.clone(),
            "scores": scores.clone(),
        })
    return runs

training_start = time.perf_counter()
text_runs = run_method(train_text_predictor)
linear_runs = run_method(train_linear_classifier)
training_seconds = time.perf_counter() - training_start
print(f"Ten head-training runs completed in {training_seconds:.1f} seconds.")


## 5. Compare the held-out results

Top-1 accuracy is the share of held-out images assigned the correct CIFAR-10 class. For the predictor, the predicted text embedding is matched to one of the ten fixed prompt embeddings. For the baseline, the largest class logit is selected.

The table shows every seed, mean, median, and a Student t 95% interval across the five independent head initializations. With five seeds, this interval is uncertain. A separate class-stratified bootstrap interval reports paired per-image differences on this fixed test sample; it does not describe training-run variation.


In [ ]:
from scipy.stats import t


def report_runs(name, runs):
    values = np.asarray([run["accuracy"] for run in runs])
    mean = values.mean()
    standard_deviation = values.std(ddof=1)
    critical_value = t.ppf(0.975, df=len(values) - 1)
    margin = critical_value * standard_deviation / np.sqrt(len(values))
    print(f"{name}")
    for run in runs:
        print(f"  seed {run['seed']:>2}: top-1 = {run['accuracy']:.3f}")
    print(f"  mean ± sample SD: {mean:.3f} ± {standard_deviation:.3f}")
    print(f"  median: {np.median(values):.3f}")
    print(f"  95% t interval across seeds (uncertain, n={len(values)}): [{mean - margin:.3f}, {mean + margin:.3f}]")
    print("  class-wise mean top-1:")
    class_means = np.mean([run["per_class_accuracy"] for run in runs], axis=0)
    for class_name, value in zip(class_names, class_means):
        print(f"    {class_name:>10}: {value:.3f}")
    return values

text_accuracies = report_runs("Text-embedding predictor", text_runs)
linear_accuracies = report_runs("Linear classifier baseline", linear_runs)

text_correct = np.stack([
    run["predictions"].eq(test_labels).numpy() for run in text_runs
]).astype(np.float64)
linear_correct = np.stack([
    run["predictions"].eq(test_labels).numpy() for run in linear_runs
]).astype(np.float64)
paired_image_differences = (text_correct - linear_correct).mean(axis=0)
labels_np = test_labels.numpy()
class_indices = [np.flatnonzero(labels_np == class_id) for class_id in range(10)]
bootstrap_rng = np.random.default_rng(314159)
bootstrap_indices = np.stack([
    bootstrap_rng.choice(indices, size=(5000, len(indices)), replace=True)
    for indices in class_indices
], axis=1)
bootstrap_differences = paired_image_differences[bootstrap_indices].mean(axis=(1, 2))
bootstrap_low, bootstrap_high = np.quantile(bootstrap_differences, [0.025, 0.975])
print(
    "Paired per-image accuracy difference (predictor - linear): "
    f"{paired_image_differences.mean():.3f}; 95% class-stratified bootstrap interval "
    f"on this fixed sample: [{bootstrap_low:.3f}, {bootstrap_high:.3f}] "
    "(5,000 resamples; not training-seed uncertainty)."
)

chance_accuracy = 1 / len(class_names)
print(f"Uniform ten-class chance reference: {chance_accuracy:.3f}")


In [ ]:
import matplotlib.pyplot as plt

means = [text_accuracies.mean(), linear_accuracies.mean()]
standard_deviations = [text_accuracies.std(ddof=1), linear_accuracies.std(ddof=1)]
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["Text embedding", "Linear classifier"], means, yerr=standard_deviations, capsize=5)
ax.axhline(chance_accuracy, color="gray", linestyle="--", label="10% chance")
ax.set_ylim(0, 1)
ax.set_ylabel("Held-out top-1 accuracy")
ax.set_title("Same frozen image features and data")
ax.legend()
plt.tight_layout()
plt.show()


## Inspect ranked text prompts

The table below shows one held-out image per class for the text-prediction run with seed 7. `candidate_scores` contains the full 1,000 × 10 score matrix for that seed. The scores are similarities to class prompts, not calibrated probabilities.


In [ ]:
example_run = next(run for run in text_runs if run["seed"] == MODEL_SEEDS[0])
example_scores = example_run["scores"]
example_predictions = example_run["predictions"]
print("true class | predicted class | top 3 prompt similarities")
for class_id, class_name in enumerate(class_names):
    sample_index = int(torch.where(test_labels == class_id)[0][0])
    top_indices = torch.topk(example_scores[sample_index], k=3).indices.tolist()
    ranking = ", ".join(
        f"{class_names[idx]} ({example_scores[sample_index, idx].item():.3f})"
        for idx in top_indices
    )
    predicted_name = class_names[int(example_predictions[sample_index])]
    print(f"{class_name:>10} | {predicted_name:>10} | {ranking}")

candidate_scores = {run["seed"]: run["scores"] for run in text_runs}
assert candidate_scores[MODEL_SEEDS[0]].shape == (1000, 10)


## What the evidence supports

The authors' paper reports a large-scale vision-language system with video classification, text-to-video retrieval, question-answering, and selective-decoding results. Those values belong to the paper and use its own models, data, compute, and benchmarks. The paper reports 1.6B parameters and describes pretraining on 2B image-text samples followed by video training.

This notebook measures a smaller task: predict embeddings for ten CIFAR-10 class prompts from frozen ImageNet ResNet-18 features, then compare classification accuracy with a linear probe. It uses 2,000 selected images, one fixed data sample, five head seeds, and no video. It does not test the paper's reported findings.

An exact reproduction still needs a verified first-party implementation and checkpoint, the paper's training data and recipes, and its evaluation benchmarks. This source review did not confirm those resources. Keep reproduction work open until they are available and independently checked.

### Next experiments

1. Replace class prompts with human-written captions from a dataset whose terms and splits are documented.
2. Compare retrieval Recall@1 with a clear image-to-text retrieval baseline.
3. Add the paper's bidirectional contrastive objective and test for representation collapse.
4. Increase data and model scale only when compute and dataset access are recorded.
